### Import needed libraries

In [76]:
import os
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical # type: ignore
from tensorflow.keras.optimizers import Adam # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import LSTM, Dense, GRU,  Masking, Dropout, Input, BatchNormalization, Layer, GlobalAveragePooling1D, Flatten # type: ignore
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, # type: ignore
                                      TensorBoard, LearningRateScheduler, )

os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/cuda'


In [77]:
X = []
y = []

data_dir = "pickles/"

for file in os.listdir(data_dir):
    file_path = os.path.join(data_dir, file)
    if os.path.exists(file_path):
        with open(file_path, "rb") as f:
            data = pickle.load(f)
        for entry in data:
            points = entry["points"]  # list of frames, each frame = list of (x,y)
            if points:
                # Flatten each frame into 1D vector
                seq = [np.array(frame, dtype=np.float32).flatten() for frame in points]
                X.append(seq)
                y.append(entry["class_name"])
    else:
        print(f"{file} not found!")

print(f"Loaded {len(X)} sequences")

Loaded 1800 sequences


In [78]:
max_seq_len = max(len(seq) for seq in X)
feature_dim = max(len(frame) for seq in X for frame in seq)  # largest frame vector size

X_padded = []
for seq in X:
    arr = np.zeros((max_seq_len, feature_dim), dtype=np.float32)
    for i, frame in enumerate(seq):
        arr[i, :len(frame)] = frame
    X_padded.append(arr)

X_padded = np.array(X_padded, dtype=np.float32)  # (num_samples, max_seq_len, feature_dim)
print("X_padded shape:", X_padded.shape)

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_onehot = to_categorical(y_encoded)

print(X_padded.shape, y_onehot.shape)
print(X_padded[0])
print(X_padded[0][0])

X_padded shape: (1800, 20, 468)
(1800, 20, 468) (1800, 6)
[[228.85187 241.72464 231.61032 ...   0.        0.        0.     ]
 [228.74829 241.68411 231.52026 ...   0.        0.        0.     ]
 [229.00572 242.69765 230.83131 ...   0.        0.        0.     ]
 ...
 [227.82431 242.60811 229.67229 ...   0.        0.        0.     ]
 [228.6836  242.5856  230.53719 ...   0.        0.        0.     ]
 [228.40543 243.42773 230.2832  ...   0.        0.        0.     ]]
[228.85187 241.72464 231.61032 243.56361 231.61032 243.56361 234.36876
 245.40257 234.36876 245.40257 238.0467  248.16103 238.0467  248.16103
 244.4831  250.      244.4831  250.      250.      250.      250.
 250.      257.35587 250.      257.35587 250.      262.87277 248.16103
 262.87277 248.16103 267.4702  245.40257 267.4702  245.40257 270.22867
 243.56361 270.22867 243.56361 272.98712 241.72464 228.85187 241.72464
 230.69083 239.88567 230.69083 239.88567 232.5298  238.0467  232.5298
 238.0467  236.20773 236.20773 236.20773 23

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y_onehot, test_size=0.2, random_state=42, shuffle=True, stratify=y_encoded
)
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.layers import Input, Masking, Bidirectional, LSTM, Dense, Dropout, Layer
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# --- Attention Layer ---
class Attention(Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="att_weight", shape=(input_shape[-1], 1),
            initializer="glorot_uniform", trainable=True
        )
        self.b = self.add_weight(
            name="att_bias", shape=(input_shape[1], 1),
            initializer="zeros", trainable=True
        )
        super(Attention, self).build(input_shape)

    def call(self, x):
        e = K.tanh(K.dot(x, self.W) + self.b)     # (batch, timesteps, 1)
        a = K.softmax(e, axis=1)                  # attention weights
        output = x * a                            # weight input sequence
        return K.sum(output, axis=1)              # context vector


# --- Model ---
def build_model(input_shape, num_classes):
    inp = Input(shape=input_shape)

    # If you know X_padded is strictly right-padded, you can remove Masking
    # Otherwise keep it, but disable cuDNN (Fix 1)
    x = Masking(mask_value=0.0)(inp)

    # Remove this line:
    # x = Masking(mask_value=0.0)(inp)

    # Then keep regular BiLSTMs (CuDNN-accelerated)
    x = Bidirectional(LSTM(64, return_sequences=True))(inp)
    x = Bidirectional(LSTM(64, return_sequences=True))(x)


    # Attention instead of pooling
    x = Attention()(x)

    # Dense layers
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.4)(x)

    out = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=inp, outputs=out)
    return model


# --- Compile and Train ---
model = build_model((X_padded.shape[1], X_padded.shape[2]), y_onehot.shape[1])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# Callbacks
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5)

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,  # <<< bumped from 8 → 32 for stabler training
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr]
)

loss, acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {acc:.3f}")


Model: "functional_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_17 (InputLayer)     │ (None, 20, 468)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_28                │ (None, 20, 128)        │       272,896 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_29                │ (None, 20, 128)        │        98,816 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_12 (Attention)        │ (None, 128)            │           148 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 389,146 (1.48 MB)

 Trainable params: 389,146 (1.48 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.4382 - loss: 1.5943 - val_accuracy: 0.7361 - val_loss: 1.3530 - learning_rate: 1.0000e-04
Epoch 2/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7479 - loss: 1.1896 - val_accuracy: 0.8278 - val_loss: 0.9996 - learning_rate: 1.0000e-04
Epoch 3/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8229 - loss: 0.8652 - val_accuracy: 0.8833 - val_loss: 0.6727 - learning_rate: 1.0000e-04
Epoch 4/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8875 - loss: 0.6092 - val_accuracy: 0.9361 - val_loss: 0.4626 - learning_rate: 1.0000e-04
Epoch 5/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9174 - loss: 0.4505 - val_accuracy: 0.9361 - val_loss: 0.3607 - learning_rate: 1.0000e-04
Epoch 6/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8840 - loss: 0.4268 - val_accuracy: 0.9389 - val_loss: 0.3153 - learning_rate: 1.0000e-04
Epoch 7/200
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.88

In [80]:
model.save("gesture_model.h5", save_format="h5")

In [81]:
# import libraries
with open('label_encoder.pkl', 'wb') as f:
  pickle.dump(le, f)

In [82]:
import tensorflow as tf
print("TF:", tf.__version__)
print("Physical GPUs:", tf.config.list_physical_devices("GPU"))
try:
    from tensorflow.python.client import device_lib
    print(device_lib.list_local_devices())
except Exception:
    pass


TF: 2.20.0
Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 8644086411273024846
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 5549522944
locality {
  bus_id: 1
  links {
  }
}
incarnation: 12757609533600026500
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6"
xla_global_id: 416903419
]


I0000 00:00:1758482712.656475    5175 gpu_device.cc:2020] Created device /device:GPU:0 with 5292 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6


In [83]:
import numpy as np
np.sum(y_train, axis=0)  # class counts


array([240., 240., 240., 240., 240., 240.])